In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import TypedDict
from dotenv import load_dotenv
import json
import re

In [ ]:
load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
)

model = ChatHuggingFace(llm=llm)

In [ ]:
class ReviewEvaluationSchema(BaseModel):
    evaluation: str = Field(description='Evaluation status: "approved" or "rejected"')
    reason: str = Field(description='Reason for the evaluation decision')

parser = JsonOutputParser(pydantic_object=ReviewEvaluationSchema)

In [ ]:
class ReviewState(TypedDict):
    review: str
    reply: str
    evaluation: str
    reason: str

In [ ]:
def generate_reply(state: ReviewState) -> ReviewState:
    review = state['review']
    prompt = f"""Write a polite and professional response to the following customer review:
Review: "{review}"
"""
    reply = model.invoke(prompt).content
    return {'reply': reply}

In [ ]:
def parse_evaluation_output(response_text: str) -> dict:
    """Robust parser to prevent OutputParserException when LLM outputs non-standard JSON."""
    # 1. Try standard JsonOutputParser
    try:
        return parser.parse(response_text)
    except Exception:
        pass

    # 2. Extract JSON using regex if enclosed in braces
    json_match = re.search(r'\{.*?\}', response_text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except Exception:
            pass

    # 3. Fallback parser for key-value outputs like `evaluation: "approved"`
    parsed = {"evaluation": "approved", "reason": "Default approval"}
    for line in response_text.strip().splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            clean_k = k.strip().lower().replace('"', '').replace("'", '')
            clean_v = v.strip().replace('"', '').replace("'", '')
            if "evaluation" in clean_k:
                parsed["evaluation"] = clean_v
            elif "reason" in clean_k:
                parsed["reason"] = clean_v

    return parsed

In [ ]:
def evaluate_reply(state: ReviewState) -> ReviewState:
    review = state['review']
    reply = state['reply']

    prompt = PromptTemplate(
        template="""You are a quality assurance reviewer. Evaluate if the reply appropriately addresses the customer review.

CRITICAL REQUIREMENT: Output MUST be a single, valid JSON object strictly matching this schema:
{format_instructions}

Do NOT output unformatted key-value text (like `evaluation: "approved"`). ALWAYS wrap in curly braces `{{"evaluation": "...", "reason": "..."}}`.

Customer Review: {review}
Generated Reply: {reply}
""",
        input_variables=["review", "reply"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )

    formatted_prompt = prompt.format(review=review, reply=reply)
    response_text = model.invoke(formatted_prompt).content
    result = parse_evaluation_output(response_text)

    return {
        'evaluation': result.get('evaluation', 'approved'),
        'reason': result.get('reason', 'Reply meets guidelines')
    }

In [ ]:
graph = StateGraph(ReviewState)

graph.add_node('generate_reply', generate_reply)
graph.add_node('evaluate_reply', evaluate_reply)

graph.add_edge(START, 'generate_reply')
graph.add_edge('generate_reply', 'evaluate_reply')
graph.add_edge('evaluate_reply', END)

workflow = graph.compile()

In [ ]:
initial_state = {
    'review': 'The product arrived on time, but the packaging was damaged and the item had minor scratches.'
}

result = workflow.invoke(initial_state)

print("=== GENERATED REPLY ===")
print(result['reply'])
print("\n=== EVALUATION STATUS ===")
print(result['evaluation'])
print("\n=== REASON ===")
print(result['reason'])